# QLoRA Fine-Tuning Pipeline for Financial MCQ

Multi-dataset, multilingual fine-tuning on:  
- **CFA** (600 questions, English)  
- **EFPA** (50 questions, Spanish)  
- **GRFinQA** (225 questions, Greek)  
- **CPA** (300 questions, Chinese)  
- **BBF** (500–1000 questions, Hindi)

Implements insights from: Bar, Phi-3, Med42, Legal/Jurisdictional, and Physics papers.

## Cell 1 — Install Required Libraries

Covers everything needed for QLoRA fine-tuning (from Bar + Phi-3 papers).

In [ ]:
# Install all required libraries for QLoRA fine-tuning (Bar + Phi-3 papers)
!pip install transformers datasets peft trl bitsandbytes accelerate
!pip install sentencepiece protobuf evaluate scikit-learn

# For multilingual tokenization (EFPA Spanish, BBF Hindi, CPA Chinese)
!pip install sacremoses langdetect sentence-transformers

## Cell 2 — Load All 5 Datasets

Task format is standardized across all datasets: **question + 4 options → label**.

Expected columns: `question`, `A`, `B`, `C`, `D`, `answer`

In [ ]:
from datasets import load_dataset, Dataset
import pandas as pd
import json

def load_mcq_dataset(path, lang):
    """Load a CSV or JSON MCQ dataset and tag with language code."""
    if path.endswith('.json') or path.endswith('.jsonl'):
        df = pd.read_json(path, lines=path.endswith('.jsonl'))
    else:
        df = pd.read_csv(path)
    # Expected columns: question, A, B, C, D, answer
    df['language'] = lang
    return df

# Update paths to point to your actual dataset files
cfa  = load_mcq_dataset('cfa_600.csv',  'en')   # 600 questions
efpa = load_mcq_dataset('efpa_50.csv',  'es')   # 50 questions
grfq = load_mcq_dataset('grfinqa.csv',  'el')   # 225 questions
cpa  = load_mcq_dataset('cpa_300.csv',  'zh')   # 300 questions
bbf  = load_mcq_dataset('bbf_500.csv',  'hi')   # 500-1000 questions

all_data = pd.concat([cfa, efpa, grfq, cpa, bbf], ignore_index=True)
print(f"Total questions: {len(all_data)}")
print(all_data['language'].value_counts())

## Cell 3 — Decontamination (Med42 Insight)

Med42 paper found that train/test overlap **falsely inflates accuracy**.  
Remove training questions that are too similar to test questions using cosine similarity.  
Uses a multilingual sentence encoder — handles all 5 languages.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
import numpy as np

# Multilingual model handles all 5 languages (en, es, el, zh, hi)
embed_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

def decontaminate(train_df, test_df, threshold=0.8):
    """
    Remove training samples that are too similar to test questions.
    Cosine similarity >= threshold → contaminated → removed.
    """
    train_q = train_df['question'].tolist()
    test_q  = test_df['question'].tolist()

    train_emb = embed_model.encode(train_q, batch_size=64, show_progress_bar=True)
    test_emb  = embed_model.encode(test_q,  batch_size=64, show_progress_bar=True)

    sims = cosine_similarity(train_emb, test_emb)
    max_sim = sims.max(axis=1)

    clean_mask = max_sim < threshold
    removed = (~clean_mask).sum()
    print(f"Removed {removed} contaminated samples ({removed / len(train_df) * 100:.1f}%)")
    return train_df[clean_mask].reset_index(drop=True)

# Stratified split to preserve language distribution, then decontaminate
train_df, test_df = train_test_split(
    all_data, test_size=0.2, stratify=all_data['language'], random_state=42
)
train_df = decontaminate(train_df, test_df)

print(f"\nTrain size: {len(train_df)}, Test size: {len(test_df)}")
print("Train language distribution:\n", train_df['language'].value_counts())

## Cell 4 — Prompt Formatting with Option Shuffling (Phi-3 Insight)

**Phi-3 paper**: Positional bias (model always picks the last option) is caused by bad prompt structure.  
Shuffling options during training prevents this.  
**Legal paper**: Output = label + full option text gives richer training signal than just "A".

In [ ]:
import random

def format_prompt(row, shuffle=True):
    """
    Alpaca-style prompt format (Phi-3 paper — best for MCQ).
    - shuffle=True: randomises option order to prevent positional bias (Phi-3 paper).
    - Output includes correct label + full option text for richer signal (Legal paper).
    """
    options = {'A': row['A'], 'B': row['B'], 'C': row['C'], 'D': row['D']}

    if shuffle:
        keys = list(options.keys())
        random.shuffle(keys)
        correct_new_idx = keys.index(row['answer'])
        new_labels = ['A', 'B', 'C', 'D']
        options_str = "\n".join([f"{new_labels[i]}. {options[keys[i]]}" for i in range(4)])
        correct_label = new_labels[correct_new_idx]
        correct_text  = options[row['answer']]
    else:
        options_str   = "\n".join([f"{k}. {v}" for k, v in options.items()])
        correct_label = row['answer']
        correct_text  = options[correct_label]

    prompt = (
        f"<|user|>\nAnswer the following financial question.\n\n"
        f"Question: {row['question']}\n\n{options_str}\n\n"
        f"Select the correct answer (A, B, C, or D).<|end|>\n"
        f"<|assistant|>"
    )
    # Richer output: label + full option text (Legal/Jurisdictional paper insight)
    output = f"{correct_label}. {correct_text}"

    return prompt + output

# Apply to training data
train_df['formatted'] = train_df.apply(format_prompt, axis=1)
print("Example formatted prompt:\n")
print(train_df['formatted'].iloc[0])

## Cell 5 — Load Model in 4-bit (QLoRA)

| Model | Size | GPU needed | Expected accuracy | Best for |
|---|---|---|---|---|
| **Phi-3.5-mini** | 3.8B | 8GB (GTX 1650) | ~90% | Low-resource, fast iteration |
| Llama-3-8B | 8B | 16GB (RTX 3080) | ~52–65% | Best baseline → fine-tune gains |
| Mistral-7B | 7B | 16GB | ~50–60% | Multilingual |
| Llama-3-13B | 13B | 24GB (A10G) | ~70%+ | Sweet spot if 24GB available |

**Recommendation**: Start with `Phi-3.5-mini` (cheapest, 90%+ on MCQ), then try Llama-3-8B for multilingual robustness.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# QLoRA config — NF4 quantisation from the affordability paper, works on single V100/A100
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # NF4 from affordability paper
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,     # double quantisation reduces memory further
)

# Change to "meta-llama/Meta-Llama-3-8B-Instruct" for Llama, etc.
MODEL_NAME = "microsoft/Phi-3.5-mini-instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print(f"Loaded: {MODEL_NAME}")
print(f"Device map: {model.hf_device_map}")

## Cell 6 — LoRA Config + Masked Loss Collator (Med42 Insight)

**Med42**: Apply LoRA to **every linear layer**, not just attention.  
**Med42 key insight**: Mask loss so the model only learns from **answer tokens**, not question tokens — prevents the model from memorising questions.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM

# Apply LoRA to every linear layer (Med42 finding — better than just q/v attention)
lora_config = LoraConfig(
    r=8,                        # rank — from Legal/Jurisdictional paper
    lora_alpha=16,              # alpha = 2× rank is standard
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",  # Med42: all linear layers, not just q/v
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Masked loss: only backprop on <|assistant|> tokens (Med42 key insight).
# The model learns HOW to answer, not to repeat questions.
response_template = "<|assistant|>"
collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template,
    tokenizer=tokenizer,
)

## Cell 7 — Training Arguments + SFTTrainer

| Hyperparameter | Value | Source |
|---|---|---|
| Learning rate | 2e-4 | Phi-3 paper |
| Batch size | 8–16 | Legal + Med42 |
| Epochs | 1–3 | Legal: 1 epoch; 60-pts paper: don't overfit |
| Max seq length | 1024 | Jurisdictional paper |
| Warmup ratio | 0.03 | Standard for LoRA |
| LR scheduler | cosine | Bar paper |

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

# Convert pandas DataFrames to HuggingFace Datasets
train_hf = Dataset.from_pandas(train_df[['formatted']])
eval_hf  = Dataset.from_pandas(test_df.assign(
    formatted=test_df.apply(lambda r: format_prompt(r, shuffle=False), axis=1)
)[['formatted']])

training_args = TrainingArguments(
    output_dir="./financial_mcq_model",
    num_train_epochs=1,                   # Legal paper: 1 epoch is enough
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,        # effective batch size = 16
    learning_rate=2e-4,                   # Phi-3 paper: best for MCQ SFT
    lr_scheduler_type="cosine",           # Bar paper
    warmup_ratio=0.03,
    max_grad_norm=0.3,
    fp16=False,
    bf16=True,                            # better for modern GPUs
    logging_steps=10,
    save_strategy="epoch",
    evaluation_strategy="steps",
    eval_steps=50,
    load_best_model_at_end=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_hf,
    eval_dataset=eval_hf,
    data_collator=collator,               # masked loss applied here
    dataset_text_field="formatted",
    max_seq_length=1024,                  # Jurisdictional paper
    args=training_args,
)

trainer.train()

## Cell 8 — Logit-Based Evaluation (Legal Paper Method)

The Jurisdictional/Legal paper found that **softmax over option logits** is more deterministic and accurate than parsing text output.  
**Physics paper insight**: Stratify results by language — basic definitions ≠ complex regulatory math.

In [ ]:
import torch
import torch.nn.functional as F

def predict_logits(model, tokenizer, row):
    """
    Legal paper method: score each option by its log-probability at the last token position.
    More reliable than parsing freeform text output for A/B/C/D.
    """
    # Resolve the token id for each option letter
    option_tokens = {
        opt: tokenizer.encode(opt, add_special_tokens=False)[0]
        for opt in ['A', 'B', 'C', 'D']
    }

    # Build prompt stopping just before the answer
    full_prompt = format_prompt(row, shuffle=False)
    prompt_only = full_prompt.split("<|assistant|>")[0] + "<|assistant|>"

    inputs = tokenizer(prompt_only, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)

    last_logits = outputs.logits[0, -1, :]   # logits at the last token position
    option_logits = {k: last_logits[v].item() for k, v in option_tokens.items()}
    prediction = max(option_logits, key=option_logits.get)
    return prediction


def evaluate_by_language(model, tokenizer, test_df):
    """
    Evaluate accuracy stratified by language (Physics paper insight).
    Your model may be expert at CFA L1 but fail at CPA accounting — track separately.
    """
    overall_preds = []
    overall_labels = []
    results = {}

    for lang in sorted(test_df['language'].unique()):
        subset = test_df[test_df['language'] == lang].reset_index(drop=True)
        preds  = [predict_logits(model, tokenizer, row) for _, row in subset.iterrows()]
        labels = subset['answer'].tolist()
        acc    = sum(p == l for p, l in zip(preds, labels)) / len(labels)
        results[lang] = round(acc * 100, 1)
        print(f"  [{lang}]  {results[lang]}%  (n={len(subset)})")
        overall_preds  += preds
        overall_labels += labels

    overall_acc = sum(p == l for p, l in zip(overall_preds, overall_labels)) / len(overall_labels)
    print(f"\n  Overall: {round(overall_acc * 100, 1)}%  (n={len(overall_labels)})")
    return results, overall_preds

print("Evaluating fine-tuned model...")
lang_results, all_preds = evaluate_by_language(model, tokenizer, test_df)

## Cell 9 — Option Bias Monitoring (Bar Paper)

**Bar paper**: Llama-2 always picked "C", Llama-3 always picked "D" before fine-tuning. SFT fixes this.  
Monitor throughout training. Ideal distribution ≈ 25% per option.  
**If bias detected** → option shuffling (Cell 4) + check training label balance below.

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

def check_option_bias(predictions, title="Prediction distribution"):
    """
    Bar paper: detect positional bias (e.g. model always picks 'C' or 'D').
    Ideal ≈ 25% each. Flag any option exceeding 40%.
    """
    counts = Counter(predictions)
    total  = len(predictions)
    print(f"{title} (ideal ≈ 25% each):")
    for opt in ['A', 'B', 'C', 'D']:
        pct  = counts.get(opt, 0) / total * 100
        bar  = "█" * int(pct / 2)
        flag = "  ← BIASED!" if pct > 40 else ""
        print(f"  {opt}: {bar} {pct:.1f}%{flag}")

    # Bar chart
    opts = ['A', 'B', 'C', 'D']
    pcts = [counts.get(o, 0) / total * 100 for o in opts]
    colors = ['red' if p > 40 else 'steelblue' for p in pcts]
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.bar(opts, pcts, color=colors)
    ax.axhline(25, color='gray', linestyle='--', label='Ideal 25%')
    ax.set_ylabel('% of predictions')
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()


def check_label_balance(df, label_col='answer'):
    """Check training label distribution; warn if any label dominates."""
    counts = df[label_col].value_counts(normalize=True) * 100
    print("Training label distribution:", counts.round(1).to_dict())
    if counts.max() > 35:
        print("⚠  Imbalanced! Consider oversampling minority labels.")
    return counts


# Run both checks
check_option_bias(all_preds, title="Post-training prediction bias")
check_label_balance(train_df)